In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, 
    roc_curve, 
    auc, 
    classification_report
)
from sklearn.preprocessing import label_binarize, LabelEncoder
from sklearn.model_selection import train_test_split
import joblib # Or pickle
import numpy as np
import shap # Make sure you have 'pip install shap'

# -------------------------------------------------------------------
# SECTION 1: DATA AND MODEL LOADING (Your Reference Script)
# -------------------------------------------------------------------

print("Loading data...")
# Make sure the path is correct from where you run the notebook
df = pd.read_excel("data/balanced_sheet1.xlsx") 

# Separate features and targets
X = df.drop(columns=["Injury Severity", "Injury Location"])
y_severity = df["Injury Severity"]
y_location = df["Injury Location"]

# Encode Injury Location
location_encoder = LabelEncoder()
y_location_encoded = location_encoder.fit_transform(y_location)

# Split datasets (same way as model training)
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X, y_severity, test_size=0.2, random_state=42)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X, y_location_encoded, test_size=0.2, random_state=42)

# Ensure numeric data for models
X_train_s = X_train_s.astype(float)
X_test_s = X_test_s.astype(float)
X_train_l = X_train_l.astype(float)
X_test_l = X_test_l.astype(float)

# --- Load your trained models ---
print("Loading models...")
model_severity = joblib.load("results/best_models/best_severity_model.pkl")
model_location = joblib.load("results/best_models/best_location_model.pkl")

# --- Get predictions ---
print("Making predictions for severity...")
y_pred_severity = model_severity.predict(X_test_s)
y_proba_severity = model_severity.predict_proba(X_test_s)

print("Making predictions for location...")
y_pred_location = model_location.predict(X_test_l)
y_proba_location = model_location.predict_proba(X_test_l)

# --- Define class names (crucial for plotting) ---
severity_classes = model_severity.classes_ 
location_classes_numeric = model_location.classes_ # These are the encoded numbers (e.g., [0, 1, 2...])
human_readable_location_classes = location_encoder.classes_ # These are the strings (e.g., ['Ankle', 'Knee'...])

print(f"Severity classes: {severity_classes}")
print(f"Location classes (numeric): {location_classes_numeric}")
print(f"Location classes (human-readable): {human_readable_location_classes}")
print("\n--- All data and models loaded. Starting visualizations. ---")


# -------------------------------------------------------------------
# SECTION 2: PLOT GENERATION
# -------------------------------------------------------------------

# --- 1. Confusion Matrix Heatmap (Severity) [Section 6.2.1] ---
print("\nGenerating Plot 1: Severity Confusion Matrix...")
try:
    cm_severity = confusion_matrix(y_test_s, y_pred_severity, labels=severity_classes)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_severity, 
        annot=True, 
        fmt='d', 
        cmap='Blues', 
        xticklabels=severity_classes, 
        yticklabels=severity_classes
    )
    plt.title('Confusion Matrix for Injury Severity (Hybrid4 Model)', fontsize=14)
    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig("confusion_matrix_severity.png", dpi=300, bbox_inches='tight')
    plt.savefig("confusion_matrix_severity.pdf", format='pdf', bbox_inches='tight')
    plt.show()
    print("Plot 1 saved as confusion_matrix_severity.png/pdf")
except Exception as e:
    print(f"Error generating Plot 1: {e}")


# --- 2. Confusion Matrix Heatmap (Location) [Section 6.2.2] ---
print("\nGenerating Plot 2: Location Confusion Matrix...")
try:
    cm_location = confusion_matrix(y_test_l, y_pred_location, labels=location_classes_numeric)
    plt.figure(figsize=(12, 10)) 
    sns.heatmap(
        cm_location, 
        annot=True, 
        fmt='d', 
        cmap='Greens', 
        # Use the numeric classes for labels, but the string names for ticks
        xticklabels=human_readable_location_classes, 
        yticklabels=human_readable_location_classes
    )
    plt.title('Confusion Matrix for Injury Location (Hybrid RF+XGB Model)', fontsize=14)
    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig("confusion_matrix_location.png", dpi=300, bbox_inches='tight')
    plt.savefig("confusion_matrix_location.pdf", format='pdf', bbox_inches='tight')
    plt.show()
    print("Plot 2 saved as confusion_matrix_location.png/pdf")
except Exception as e:
    print(f"Error generating Plot 2: {e}")


# --- 3. Class-Specific F1-Score Bar Chart (Location) [Section 6.2.2] ---
print("\nGenerating Plot 3: Location F1-Score Bar Chart...")
try:
    report_dict = classification_report(
        y_test_l, 
        y_pred_location, 
        labels=location_classes_numeric, 
        target_names=human_readable_location_classes,
        output_dict=True
    )
    
    report_df = pd.DataFrame(report_dict).transpose()
    # Keep only the classes, remove averages
    report_df = report_df.loc[human_readable_location_classes] 
    report_df.reset_index(inplace=True)
    report_df.rename(columns={'index': 'Location'}, inplace=True)

    plt.figure(figsize=(10, 8))
    sns.barplot(
        data=report_df.sort_values('f1-score', ascending=False), 
        x='f1-score', 
        y='Location',
        palette='viridis'
    )
    plt.title('F1-Score per Injury Location (Hybrid RF+XGB)', fontsize=14)
    plt.xlabel('F1-Score', fontsize=12)
    plt.ylabel('Anatomical Location', fontsize=12)
    plt.xlim(0, 1.0)
    plt.tight_layout()
    plt.savefig("f1_scores_location.png", dpi=300, bbox_inches='tight')
    plt.savefig("f1_scores_location.pdf", format='pdf', bbox_inches='tight')
    plt.show()
    print("Plot 3 saved as f1_scores_location.png/pdf")
except Exception as e:
    print(f"Error generating Plot 3: {e}")


# --- 4. ROC Curve + AUC (Severity) [Section 6.2.1] ---
print("\nGenerating Plot 4: Severity ROC Curve...")
try:
    y_test_severity_bin = label_binarize(y_test_s, classes=severity_classes)
    n_classes_severity = len(severity_classes)

    fpr, tpr, roc_auc = dict(), dict(), dict()
    for i in range(n_classes_severity):
        fpr[i], tpr[i], _ = roc_curve(y_test_severity_bin[:, i], y_proba_severity[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure(figsize=(10, 8))
    colors = ['blue', 'red', 'green', 'orange', 'purple'] # Add more if needed
    for i, color in zip(range(n_classes_severity), colors):
        plt.plot(
            fpr[i], 
            tpr[i], 
            color=color, 
            lw=2, 
            label=f'ROC curve for class {severity_classes[i]} (AUC = {roc_auc[i]:.2f})'
        )
    
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('Multi-Class ROC Curve for Injury Severity (Hybrid4)', fontsize=14)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig("roc_curve_severity.png", dpi=300, bbox_inches='tight')
    plt.savefig("roc_curve_severity.pdf", format='pdf', bbox_inches='tight')
    plt.show()
    print("Plot 4 saved as roc_curve_severity.png/pdf")
except Exception as e:
    print(f"Error generating Plot 4: {e}")


# --- 5. ROC Curve + AUC (Location) [Section 6.2.2] ---
print("\nGenerating Plot 5: Location ROC Curve...")
try:
    y_test_location_bin = label_binarize(y_test_l, classes=location_classes_numeric)
    n_classes_location = len(location_classes_numeric)

    fpr_loc, tpr_loc, roc_auc_loc = dict(), dict(), dict()
    for i in range(n_classes_location):
        fpr_loc[i], tpr_loc[i], _ = roc_curve(y_test_location_bin[:, i], y_proba_location[:, i])
        roc_auc_loc[i] = auc(fpr_loc[i], tpr_loc[i])

    plt.figure(figsize=(12, 10))
    colors = plt.cm.get_cmap('tab20', n_classes_location)

    for i in range(n_classes_location):
        plt.plot(
            fpr_loc[i], 
            tpr_loc[i], 
            color=colors(i), 
            lw=2, 
            # Use the human-readable names for the legend
            label=f'{human_readable_location_classes[i]} (AUC = {roc_auc_loc[i]:.2f})'
        )

    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('Multi-Class ROC Curve for Injury Location (Hybrid RF+XGB)', fontsize=14)
    plt.legend(loc="lower right", fontsize='small')
    plt.tight_layout()
    plt.savefig("roc_curve_location.png", dpi=300, bbox_inches='tight')
    plt.savefig("roc_curve_location.pdf", format='pdf', bbox_inches='tight')
    plt.show()
    print("Plot 5 saved as roc_curve_location.png/pdf")
except Exception as e:
    print(f"Error generating Plot 5: {e}")


# --- 6. EDA Plots (Histograms & Correlation) [Chapter 3] ---
print("\nGenerating Plot 6: EDA Plots...")
try:
    # Use the original full dataframe 'df' for EDA
    
    # 6a. Histograms
    key_features_eda = ['Weekly Training Hours', 'Trunk Flexion (cm)', 'Stick Test (cm)', 'BMI']
    plt.figure(figsize=(12, 10))
    for i, feature in enumerate(key_features_eda, 1):
        plt.subplot(2, 2, i)
        sns.histplot(df[feature], kde=True, bins=20)
        plt.title(f'Distribution of {feature}', fontsize=12)
    plt.tight_layout()
    plt.savefig("eda_histograms.png", dpi=300, bbox_inches='tight')
    plt.savefig("eda_histograms.pdf", format='pdf', bbox_inches='tight')
    plt.show()
    print("Plot 6a (Histograms) saved.")

    # 6b. Correlation Heatmap
    numerical_features_df = df.select_dtypes(include=[np.number])
    plt.figure(figsize=(12, 10))
    corr_matrix = numerical_features_df.corr()
    sns.heatmap(
        corr_matrix, 
        annot=True, 
        cmap='coolwarm', 
        fmt='.2f', 
        linewidths=0.5
    )
    plt.title('Feature Correlation Heatmap', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig("eda_correlation_heatmap.png", dpi=300, bbox_inches='tight')
    plt.savefig("eda_correlation_heatmap.pdf", format='pdf', bbox_inches='tight')
    plt.show()
    print("Plot 6b (Correlation Heatmap) saved.")
except Exception as e:
    print(f"Error generating Plot 6: {e}")


# -------------------------------------------------------------------
# SECTION 3: SHAP PLOTS (Model Interpretation)
# -------------------------------------------------------------------
# This is the most computationally expensive part.
# We will explain the SEVERITY model, focusing on the "Severe" class.

print("\nGenerating Plot 7: SHAP Plots (this may take a few minutes)...")
try:
    # 1. Create a summary background dataset from training data
    # We use shap.kmeans to create a weighted summary (e.g., 100 points)
    X_train_s_summary = shap.kmeans(X_train_s, 100)

    # 2. Find the index for the 'Severe' class
    # This is the class we want to explain
    severe_class_index = list(severity_classes).index('Severe')
    print(f"Explaining the 'Severe' class at index: {severe_class_index}")

    # 3. Define a prediction function for the explainer
    # It must take a numpy array and return a 1D array of probabilities
    # for our chosen class ('Severe')
    def predict_fn_severe(x):
        # 1. Convert numpy array back to DataFrame with correct feature names
        x_df = pd.DataFrame(x, columns=X.columns)
        # 2. Get probabilities from the model
        probas = model_severity.predict_proba(x_df)
        # 3. Return only the probabilities for the 'Severe' class
        return probas[:, severe_class_index]

    # 4. Initialize the KernelExplainer
    explainer_severity = shap.KernelExplainer(predict_fn_severe, X_train_s_summary)

    # 5. Calculate SHAP values
    # We'll use a subset of the test set (e.g., 50 samples) for speed.
    # For your final paper, you might run it on all of X_test_s if you have time.
    X_test_s_sample = X_test_s.iloc[0:50]
    print(f"Calculating SHAP values for {len(X_test_s_sample)} samples...")
    shap_values_severity = explainer_severity.shap_values(X_test_s_sample)
    
    # --- 7a. SHAP Summary Plot [Section 6.3.1] ---
    print("Generating SHAP Summary Plot...")
    shap.summary_plot(
        shap_values_severity, 
        X_test_s_sample,
        feature_names=X.columns,
        show=False
    )
    plt.title('SHAP Summary Plot for "Severe" Injury Risk (Hybrid4)', fontsize=14)
    plt.savefig("shap_summary_plot_severity.png", dpi=300, bbox_inches='tight')
    plt.savefig("shap_summary_plot_severity.pdf", format='pdf', bbox_inches='tight')
    plt.show()
    print("Plot 7a (SHAP Summary) saved.")

    # --- 7b. SHAP Force Plot (High-Risk Example) [Section 6.3.1] ---
    print("Generating SHAP Force Plot (High-Risk Example)...")
    # Find an athlete in our sample with a high SHAP value (high risk)
    high_risk_index = np.argmax(shap_values_severity)
    
    # We need to use shap.force_plot with matplotlib=True to save it
    # This requires a slightly different setup
    force_plot_high = shap.force_plot(
        explainer_severity.expected_value, 
        shap_values_severity[high_risk_index, :], 
        X_test_s_sample.iloc[high_risk_index, :],
        feature_names=X.columns,
        matplotlib=True,
        show=False
    )
    force_plot_high.savefig("shap_force_plot_high_risk.png", dpi=300, bbox_inches='tight')
    # Note: To view this in the notebook, run the line below in a new cell:
    # shap.initjs()
    # shap.force_plot(explainer_severity.expected_value, shap_values_severity[high_risk_index, :], X_test_s_sample.iloc[high_risk_index, :], feature_names=X.columns)
    print("Plot 7b (SHAP High Risk) saved.")


    # --- 7c. SHAP Force Plot (Low-Risk Example) [Section 6.3.1] ---
    print("Generating SHAP Force Plot (Low-Risk Example)...")
    # Find an athlete in our sample with a low SHAP value (low risk)
    low_risk_index = np.argmin(shap_values_severity)

    force_plot_low = shap.force_plot(
        explainer_severity.expected_value, 
        shap_values_severity[low_risk_index, :], 
        X_test_s_sample.iloc[low_risk_index, :],
        feature_names=X.columns,
        matplotlib=True,
        show=False
    )
    force_plot_low.savefig("shap_force_plot_low_risk.png", dpi=300, bbox_inches='tight')
    print("Plot 7c (SHAP Low Risk) saved.")

except Exception as e:
    print(f"Error generating Plot 7 (SHAP): {e}")


print("\n--- All visualizations generated and saved! ---")

Loading data...


FileNotFoundError: [Errno 2] No such file or directory: 'data/balanced_sheet1.xlsx'